# code for calculating researcher needs

In [1]:
import pandas
import pygsheets
import numpy

In [20]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')

#spreadsheet[1] "Gas Pipelines" tab is the second index
gas_pipes = spreadsheet.worksheet('title','Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

gas_pipes = gas_pipes.replace('--', numpy.nan)
gas_pipes = gas_pipes[gas_pipes['PipelineName']!='']

gas_pipes = gas_pipes.replace('--', numpy.nan)
gas_pipes = gas_pipes[gas_pipes['PipelineName']!='']

pipes_df_orig = pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

#get country ratios sheet
country_ratios_df = spreadsheet.worksheet('title', 'Country ratios by pipeline').get_as_df()

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_22816/766882344.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gas_pipes = gas_pipes.replace('--', numpy.nan)


In [21]:
status_list = ['proposed', 'construction', 'shelved', 'cancelled', 
               'operating', 'idle', 'mothballed', 'retired']
country_list = sorted(list(set(country_ratios_df['Country'])))
region_list = sorted(list(set(country_ratios_df['Region'])))

## how many oil pipelines per region?

In [22]:
projectid_count_df = oil_pipes.groupby(['Status','StartSubRegion'])['ProjectID'].count()
#projectid_count_df = pandas.DataFrame(pipes_df_orig.loc[pipes_df_orig.Fuel.isin(['Gas','Hydrogen'])].groupby(['Status','StartSubRegion'])['ProjectID'].count())

projectid_count_df.unstack().transpose().replace(numpy.nan,0)

Status,,N/A,cancelled,construction,idle,mothballed,operating,proposed,retired,shelved
StartSubRegion,,,,,,,,,,
--,7.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0
Australia and New Zealand,0.0,0.0,0.0,0.0,0.0,0.0,21.0,0.0,3.0,0.0
Central Asia,0.0,0.0,1.0,0.0,0.0,2.0,33.0,5.0,0.0,0.0
Eastern Asia,0.0,1.0,7.0,23.0,1.0,0.0,239.0,67.0,25.0,1.0
Eastern Europe,0.0,0.0,9.0,11.0,11.0,3.0,146.0,11.0,3.0,3.0
Latin America and the Caribbean,0.0,0.0,3.0,2.0,1.0,0.0,158.0,16.0,0.0,1.0
Melanesia,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
Northern Africa,0.0,0.0,1.0,14.0,0.0,0.0,117.0,5.0,0.0,1.0
Northern America,0.0,1.0,56.0,12.0,0.0,2.0,387.0,23.0,4.0,4.0


## how many gas pipelines per region

In [23]:
projectid_count_df = gas_pipes.groupby(['Status','StartSubRegion'])['ProjectID'].count()

projectid_count_df.unstack().transpose().replace(numpy.nan,0)

Status,N/A,cancelled,construction,idle,mixed status,mothballed,operating,proposed,retired,shelved
StartSubRegion,,,,,,,,,,
Australia and New Zealand,2.0,30.0,1.0,3.0,0.0,1.0,97.0,16.0,0.0,3.0
Central Asia,0.0,1.0,5.0,0.0,0.0,0.0,59.0,11.0,0.0,1.0
Eastern Asia,2.0,39.0,115.0,0.0,1.0,0.0,724.0,183.0,3.0,35.0
Eastern Europe,0.0,18.0,21.0,1.0,0.0,5.0,322.0,82.0,4.0,4.0
Latin America and the Caribbean,0.0,14.0,6.0,1.0,0.0,1.0,160.0,62.0,0.0,12.0
Melanesia,0.0,2.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0
Northern Africa,0.0,2.0,15.0,0.0,0.0,0.0,208.0,5.0,0.0,1.0
Northern America,1.0,75.0,30.0,2.0,0.0,0.0,363.0,102.0,0.0,12.0
Northern Europe,0.0,1.0,2.0,0.0,0.0,0.0,114.0,6.0,5.0,2.0


## print all unique country names

In [24]:
countries_list = pipes_df_orig['CountriesOrAreas'].str.split(', ').tolist()
countries_list_flat = sorted(list(set([item for items in countries_list for item in items])))

In [27]:
mask = pipes_df_orig['CountriesOrAreas'].str.endswith(' ')
pipes_df_orig.loc[mask, ['ProjectID','PipelineName','CountriesOrAreas']]

,ProjectID,PipelineName,CountriesOrAreas


In [28]:
for i in countries_list_flat:
    print(i)


Afghanistan
Albania
Algeria
Angola
Argentina
Armenia
Australia
Austria
Azerbaijan
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Benin
Bolivia
Bosnia and Herzegovina
Brazil
Brunei
Bulgaria
Cambodia
Cameroon
Canada
Chad
Chile
China
Colombia
Croatia
Cuba
Cyprus
Czech Republic
Côte d'Ivoire
Denmark
Djibouti
Dominican Republic
Ecuador
Egypt
Egypt,Cyprus
El Salvador
Equatorial Guinea
Estonia
Ethiopia
Finland
France
Gabon
Georgia
Germany
Ghana
Greece
Grenada
Guadeloupe
Guatemala
Guinea
Guinea-Bissau
Guyana
Honduras
Hong Kong
Hungary
Iceland
India
Indonesia
Iran
Iraq
Ireland
Israel
Italy
Japan
Joint Petroleum Development Area
Jordan
Kazakhstan
Kazkahstan
Kazkakhstan
Kenya
Kosovo
Kuwait
Kyrgyzstan
Laos
Latvia
Lebanon
Liberia
Libya
Lithuania
Luxembourg
Malaysia
Malta
Martinique
Mauritania
Mexico
Moldova
Mongolia
Montenegro
Morocco
Mozambique
Myanmar
Namibia
Nepal
Netherlands
New Zealand
Niger
Nigeria
North Korea
North Macedonia
Norway
Oman
Pakistan
Palestine
Panama
Papua New Guinea
Paraguay
Peru


## country-specific calculations

### for Nagwa or Arabic language expert

In [33]:
arabic_language_countries = ['Algeria', 
                             'Bahrain', 
                           #'Chad',  # removed during 2026 allocation because French is dominant
                           'Comoros', 
                           'Djibouti', 
                           'Egypt', 
                           'Iraq', 
                           'Jordan', 
                           'Kuwait', 
                           'Lebanon', 
                           'Libya', 
                           'Mauritania', 
                           'Morocco', 
                           'Oman', 
                           'Qatar', 
                           'Saudi Arabia', 
                           'Somalia', 
                           'Sudan', 
                           'Syria', 
                           'Tunisia', 
                           'United Arab Emirates', 
                           'Yemen']

## Russian speaking countries

In [34]:
russian_language_countries = ['Russia', 
                              'Belarus', 
                              'Kyrgyzstan',
                              'Kazakhstan', 
                              'Ukraine', 
                              'Azerbaijan', 
                              'Estonia', 
                              'Georgia', 
                              'Latvia', 
                              'Lithuania', 
                              'Moldova', 
                              'Tajikistan', 
                              'Turkmenistan', 
                              'Uzbekistan',
                              'Serbia',
                              'Bulgaria',
                              'Georgia',
                              'Armenia'
                              'Estonia',
                              'Latvia',
                              'Lithuania'
                             ]

# includes baltics

## Asia gas tracker

Warda has done AGT countries plus Canada and Japan at times

In [35]:
# asia_gas_tracker_countries = [ \
#                                 'Bangladesh',
#                                 'Brunei',
#                                 'Cambodia',
#                                 #'China',
#                                 #'Hong Kong',
#                                 'India',
#                                 'Indonesia',
#                                 'Japan',
#                                 #'Macao',
#                                 'Malaysia',
#                                 'Myanmar',
#                                 'Pakistan',
#                                 'Philippines',
#                                 'Singapore',
#                                 'South Korea',
#                                 'Sri Lanka',
#                                 'Taiwan',
#                                 'Thailand',
#                                 'Vietnam'
#                              ]

# below updated Sep 2023
asia_gas_tracker_countries = [ \
'Afghanistan',
'Bangladesh',
'Bhutan',
'Brunei',
'Cambodia',
#'China',
#'Hong Kong',
'India',
'Indonesia',
'Iran',
'Japan',
'Laos',
#'Macao',
'Malaysia',
'Mongolia',
'Myanmar',
'Nepal',
'North Korea',
'Pakistan',
'Philippines',
'Singapore',
'South Korea',
'Sri Lanka',
'Taiwan',
'Thailand',
'Timor-Leste',
'Vietnam'
]

## for oil

## arabic

In [37]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(arabic_language_countries))].shape

(404, 110)

In [38]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(arabic_language_countries))&
                  (oil_pipes.Status.isin(['proposed','construction','shelved']))].shape

(56, 110)

### Russian speaking countries

In [39]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(russian_language_countries))].shape

(246, 110)

In [40]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(russian_language_countries))&
                  (oil_pipes.Status.isin(['proposed','construction','shelved']))].shape

(31, 110)

## asia gas tracker

In [41]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(asia_gas_tracker_countries))].shape

(148, 110)

In [42]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(asia_gas_tracker_countries))&
                  (oil_pipes.Status.isin(['proposed','construction','shelved']))].shape

(22, 110)

### China/Taiwan/Hong Kong/Macao

In [43]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(['China','Taiwan','Hong Kong','Macao']))].shape

(361, 110)

In [47]:
oil_pipes.loc[(oil_pipes.StartCountryOrArea.isin(['China','Taiwan','Hong Kong','Macao']))&
                  (oil_pipes.Status.isin(['proposed','construction','shelved']))].shape

(91, 110)

# for gas

In [50]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(arabic_language_countries))].shape

(477, 143)

In [51]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(arabic_language_countries))&
                  (gas_pipes.Status.isin(['proposed','construction','shelved']))].shape

(67, 143)

### Russian speaking countries

In [52]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(russian_language_countries))].shape

(490, 143)

In [53]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(russian_language_countries))&
                  (gas_pipes.Status.isin(['proposed','construction','shelved']))].shape

(124, 143)

## asia gas tracker

In [54]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(asia_gas_tracker_countries))].shape

(493, 143)

In [55]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(asia_gas_tracker_countries))&
                  (gas_pipes.Status.isin(['proposed','construction','shelved']))].shape

(77, 143)

### China/Taiwan/Hong Kong/Macao

In [56]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(['China','Taiwan','Hong Kong','Macao']))].shape

(984, 143)

In [57]:
gas_pipes.loc[(gas_pipes.StartCountryOrArea.isin(['China','Taiwan','Hong Kong','Macao']))&
                  (gas_pipes.Status.isin(['proposed','construction','shelved']))].shape

(332, 143)